In [2]:
import ee
ee.Authenticate()
ee.Initialize(project='gee-research123456')

# ============================================================
# 1️⃣ STUDY AREA (TEKEZE BASIN)
# ============================================================
aoi = ee.FeatureCollection(
    'projects/gee-research123456/assets/tekeze'
).geometry()

# ============================================================
# 2️⃣ LOAD ERA5-LAND DATA
# ============================================================
era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
        .filterBounds(aoi))

# ============================================================
# 3️⃣ STUDY PERIOD
# ============================================================
startYear = 1981
endYear = 1999

outFolder = 'ERA5_Tmean_Monthly_tekeze'

# ============================================================
# 4️⃣ DAILY TO MEAN TEMPERATURE FUNCTION
# ============================================================
def daily_tmean(img):
    """
    ERA5 temperature is in Kelvin → convert to Celsius
    """
    tmean = img.select('temperature_2m').subtract(273.15)
    return tmean.rename('tmean')

# ============================================================
# 5️⃣ MONTHLY AGGREGATION FUNCTION
# ============================================================
def monthly_tmean(year, month):

    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    img = (era5
           .filterDate(start, end)
           .map(daily_tmean)
           .mean()
           .clip(aoi)
           .set('system:time_start', start.millis()))

    return img

# ============================================================
# 6️⃣ EXPORT LOOP
# ============================================================
for year in range(startYear, endYear + 1):
    for month in range(1, 13):

        img = monthly_tmean(year, month)

        date = f"{year}{month:02d}"

        task = ee.batch.Export.image.toDrive(
            image=img,
            description='ERA5_Tmean_' + date,
            folder=outFolder,
            fileNamePrefix='ERA5_Tmean_' + date,
            region=aoi,
            scale=5000,   # match CHIRPS scale for CNN consistency
            crs='EPSG:4326',
            maxPixels=1e13
        )

        task.start()
        print("✅ ERA5 export started:", date)

print("✔ All ERA5 temperature exports submitted.")

✅ ERA5 export started: 198101
✅ ERA5 export started: 198102
✅ ERA5 export started: 198103
✅ ERA5 export started: 198104
✅ ERA5 export started: 198105
✅ ERA5 export started: 198106
✅ ERA5 export started: 198107
✅ ERA5 export started: 198108
✅ ERA5 export started: 198109
✅ ERA5 export started: 198110
✅ ERA5 export started: 198111
✅ ERA5 export started: 198112
✅ ERA5 export started: 198201
✅ ERA5 export started: 198202
✅ ERA5 export started: 198203
✅ ERA5 export started: 198204
✅ ERA5 export started: 198205
✅ ERA5 export started: 198206
✅ ERA5 export started: 198207
✅ ERA5 export started: 198208
✅ ERA5 export started: 198209
✅ ERA5 export started: 198210
✅ ERA5 export started: 198211
✅ ERA5 export started: 198212
✅ ERA5 export started: 198301
✅ ERA5 export started: 198302
✅ ERA5 export started: 198303
✅ ERA5 export started: 198304
✅ ERA5 export started: 198305
✅ ERA5 export started: 198306
✅ ERA5 export started: 198307
✅ ERA5 export started: 198308
✅ ERA5 export started: 198309
✅ ERA5 exp

In [ ]:
import ee

ee.Authenticate(force=True)
ee.Initialize(project='tekeze-drought-457823')  # paste your exact project ID

print("✅ Done:", ee.data.getAssetRoots())